# D2 Forestry Management Workflow Notebook

Sectioned notebook with API connectors + fallback, simple and complex tracks, and exportable results.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt

OUTPUT_DIR = Path.cwd() / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALLOW_LIVE_API = os.getenv('GEOPROMPT_ALLOW_LIVE_API', '0') == '1'


def fetch_json(url: str, fallback: dict) -> dict:
    if not ALLOW_LIVE_API:
        return fallback
    try:
        req = Request(url, headers={'User-Agent': 'geoprompt-section-d-notebook/2.0'})
        with urlopen(req, timeout=10) as response:  # nosec B310
            return json.loads(response.read().decode('utf-8'))
    except (URLError, TimeoutError, ValueError):
        return fallback


import geoprompt as gp
from geoprompt import GeoPromptFrame, frame_to_geojson, write_geojson
from geoprompt.tools import build_scenario_report, export_scenario_report

## Section A: Pull Forestry Data Sources


In [2]:
fia = fetch_json('https://apps.fs.usda.gov/fiadb-api/fullreport', {'rows': []})
firms = fetch_json('https://firms.modaps.eosdis.nasa.gov/api/area/csv', {'rows': []})
noaa = fetch_json('https://api.weather.gov/points/44.05,-121.31', {'properties': {'forecast': None}})
print('FIA rows (fallback if offline):', len(fia.get('rows', [])))
print('FIRMS rows (fallback if offline):', len(firms.get('rows', [])))
print('NOAA forecast pointer exists:', bool(noaa.get('properties', {}).get('forecast')))


FIA rows (fallback if offline): 0
FIRMS rows (fallback if offline): 0
NOAA forecast pointer exists: False


## Section B: Simple Track


In [ ]:
RAW_STANDS = [
    {'stand_id': 'S1', 'fuel_load': 0.72, 'slope': 0.35, 'distance_to_assets_km': 0.8,
     'geometry': {'type': 'Point', 'coordinates': [-121.6, 44.3]}},
    {'stand_id': 'S2', 'fuel_load': 0.63, 'slope': 0.25, 'distance_to_assets_km': 1.4,
     'geometry': {'type': 'Point', 'coordinates': [-121.2, 44.1]}},
    {'stand_id': 'S3', 'fuel_load': 0.81, 'slope': 0.41, 'distance_to_assets_km': 0.6,
     'geometry': {'type': 'Point', 'coordinates': [-121.0, 44.4]}},
    {'stand_id': 'S4', 'fuel_load': 0.77, 'slope': 0.39, 'distance_to_assets_km': 0.9,
     'geometry': {'type': 'Point', 'coordinates': [-120.8, 44.2]}},
    {'stand_id': 'S5', 'fuel_load': 0.58, 'slope': 0.21, 'distance_to_assets_km': 1.8,
     'geometry': {'type': 'Point', 'coordinates': [-121.4, 43.9]}},
]

# Build GeoPromptFrame — this is the native spatial unit for GeoPrompt.
stands_frame = GeoPromptFrame(RAW_STANDS, geometry_column='geometry', crs='EPSG:4326')

# Compute risk scores using GeoPromptFrame records (plain dicts — no Pylance Scalar issues).
enriched_rows = []
for row in stands_frame.to_records():
    fuel = float(row['fuel_load'])
    slope = float(row['slope'])
    dist = float(row['distance_to_assets_km'])
    risk = round(fuel * 0.5 + slope * 0.3 + (1.0 / dist) * 0.2, 4)
    enriched_rows.append({**row, 'risk_score': risk})

# Sort by risk descending and assign priority ranks.
enriched_rows.sort(key=lambda r: -float(r['risk_score']))
for rank, row in enumerate(enriched_rows, start=1):
    row['priority_rank'] = rank

stands_frame = GeoPromptFrame(enriched_rows, geometry_column='geometry', crs='EPSG:4326')

# Spatial analysis: nearest neighbor distances between stands.
neighbors = stands_frame.nearest_neighbors(id_column='stand_id', k=1, distance_method='haversine')
print('--- Stand summary ---')
print(json.dumps(stands_frame.summary(), indent=2, default=str))
print('\n--- Nearest neighbor pairs ---')
for nb in neighbors:
    print(f"  {nb['origin']} → {nb['neighbor']}  dist={nb['distance']:.5f}°")

# Write map GeoJSON using GeoPrompt.
map_geojson_path = OUTPUT_DIR / 'd2-notebook-priority-map.geojson'
write_geojson(map_geojson_path, stands_frame)
print(f'\nWrote GeoJSON map: {map_geojson_path}')

# ── Inline scatter map ──────────────────────────────────────────────────────
records = stands_frame.to_records()
lons = [float(r['geometry']['coordinates'][0]) for r in records]
lats = [float(r['geometry']['coordinates'][1]) for r in records]
risks = [float(r['risk_score']) for r in records]
labels = [str(r['stand_id']) for r in records]
ranks = [int(r['priority_rank']) for r in records]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(lons, lats, c=risks, cmap='YlOrRd', s=160, edgecolors='#333', zorder=5)
for lon, lat, label, rank in zip(lons, lats, labels, ranks):
    axes[0].annotate(f'{label}\n(#{rank})', (lon, lat), textcoords='offset points',
                     xytext=(5, 5), fontsize=8)
plt.colorbar(sc, ax=axes[0], label='Risk Score')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('D2 Forestry Stand Risk Map')
axes[0].grid(True, alpha=0.3)

# Bar chart: risk score by stand ranked
sorted_records = sorted(records, key=lambda r: int(r['priority_rank']))
bar_labels = [str(r['stand_id']) for r in sorted_records]
bar_values = [float(r['risk_score']) for r in sorted_records]
bar_colors = ['#c0392b' if v > 0.75 else '#e67e22' if v > 0.6 else '#27ae60' for v in bar_values]
axes[1].barh(bar_labels, bar_values, color=bar_colors)
axes[1].axvline(0.65, color='#555', linestyle='--', alpha=0.7, label='Alert threshold')
axes[1].set_xlabel('Risk Score')
axes[1].set_title('Stand Risk Score Ranking')
axes[1].legend()
axes[1].grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Save JSON summary.
simple_payload = {
    'track': 'simple',
    'stands': records,
    'priority_map_geojson': str(map_geojson_path),
}
(OUTPUT_DIR / 'd2-notebook-simple.json').write_text(json.dumps(simple_payload, indent=2, default=str), encoding='utf-8')
print('Wrote d2-notebook-simple.json')

GeoPrompt map feature count: 5


,stand_id,fuel_load,slope,distance_to_assets_km,lon,lat,risk_score,priority_rank
2,S3,0.81,0.41,0.6,-121.0,44.4,0.8613,1
3,S4,0.77,0.39,0.9,-120.8,44.2,0.7242,2
0,S1,0.72,0.35,0.8,-121.6,44.3,0.7150,3
1,S2,0.63,0.25,1.4,-121.2,44.1,0.5329,4
4,S5,0.58,0.21,1.8,-121.4,43.9,0.4641,5


Wrote D:\Github\geoprompt\outputs\d2-notebook-simple.json
Wrote D:\Github\geoprompt\outputs\d2-notebook-priority-map.geojson
Wrote D:\Github\geoprompt\outputs\d2-notebook-priority-map.html


## Section C: Complex Track


In [ ]:
scenarios = {
    'no_action':       {'risk': 0.74, 'cost_musd': 0.0,  'expected_loss_musd': 240.0},
    'thinning':        {'risk': 0.51, 'cost_musd': 65.0, 'expected_loss_musd': 170.0},
    'prescribed_burn': {'risk': 0.43, 'cost_musd': 92.0, 'expected_loss_musd': 135.0},
}

report = build_scenario_report(
    scenarios['no_action'], scenarios['prescribed_burn'],
    higher_is_better=[],
)
report_path = export_scenario_report(report, OUTPUT_DIR / 'd2-notebook-scenario-report.json')
print('Scenario report:', report_path)

# Compute scenario scores.
scenario_records = []
for name, vals in scenarios.items():
    score = round(
        (1 - vals['risk']) * 0.5
        + (1.0 / max(vals['cost_musd'] + 1, 1)) * 10 * 0.2
        + (1.0 / vals['expected_loss_musd']) * 50 * 0.3,
        4,
    )
    scenario_records.append({'scenario': name, **vals, 'score': score})
scenario_records.sort(key=lambda r: -float(r['score']))

# ── Inline scenario comparison chart ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

names = [r['scenario'] for r in scenario_records]
colors = ['#27ae60', '#e67e22', '#c0392b']

axes[0].barh(names, [r['risk'] for r in scenario_records], color=colors)
axes[0].set_xlabel('Fire Risk')
axes[0].set_title('Risk by Scenario')
axes[0].grid(True, axis='x', alpha=0.3)

axes[1].barh(names, [r['expected_loss_musd'] for r in scenario_records], color=colors)
axes[1].set_xlabel('Expected Loss ($M USD)')
axes[1].set_title('Expected Loss by Scenario')
axes[1].grid(True, axis='x', alpha=0.3)

axes[2].barh(names, [r['score'] for r in scenario_records], color=colors)
axes[2].set_xlabel('Composite Score')
axes[2].set_title('Composite Score (higher = better)')
axes[2].grid(True, axis='x', alpha=0.3)

plt.suptitle('D2 Forestry — Scenario Comparison', fontweight='bold')
plt.tight_layout()
plt.show()

complex_payload = {
    'track': 'complex',
    'scenario_ranking': scenario_records,
    'scenario_report_path': str(report_path),
}
(OUTPUT_DIR / 'd2-notebook-complex.json').write_text(json.dumps(complex_payload, indent=2, default=str), encoding='utf-8')
print('Wrote d2-notebook-complex.json')

,scenario,risk,cost,expected_loss,score
0,no_action,0.74,0.0,240.0,2.1925
2,prescribed_burn,0.43,92.0,135.0,0.4176
1,thinning,0.51,65.0,170.0,0.3635


Wrote D:\Github\geoprompt\outputs\d2-notebook-complex.json
